# Tech Challenge Fase 2

## Sprint 2 — Exploração da Camada Bronze

## Objetivo

Este notebook realiza a análise exploratória dos datasets da camada Bronze.

A partir dessa análise são identificadas:

- qualidade dos dados;
- chaves candidatas;
- valores nulos;
- duplicidades;
- oportunidades de transformação;

As evidências obtidas servirão como base para a construção da camada Silver.

## Estrutura do notebook

1. Carregamento dos datasets

2. Funções auxiliares

3. Exploração da tabela UF

4. Exploração da tabela Município

5. Exploração da tabela Meta Brasil

6. Exploração da tabela Meta UF

7. Exploração da tabela Meta Município

8. Exploração da tabela Alunos

9. Conclusões gerais

In [22]:
from pathlib import Path

import pandas as pd

In [23]:
BASE_DIR = Path.cwd().parent

BRONZE_DIR = BASE_DIR / "data" / "bronze"

BRONZE_DIR

PosixPath('/home/erick/Documents/FIAP/fase_2/tech_challenge/projeto/tech_challenge_2_alfabetizacao/data/bronze')

In [18]:
datasets = {}

for arquivo in BRONZE_DIR.glob("*.parquet"):
    datasets[arquivo.stem] = pd.read_parquet(arquivo)

print(f"{len(datasets)} datasets carregados.")

6 datasets carregados.


In [19]:
for nome, df in datasets.items():
    print(f"{nome:35} {df.shape[0]:>10,} linhas  {df.shape[1]:>3} colunas")

meta_alfabetizacao_uf                       81 linhas   13 colunas
uf                                         145 linhas   16 colunas
municipio                               23,995 linhas   16 colunas
alunos                               3,867,999 linhas   13 colunas
meta_alfabetizacao_brasil                    3 linhas   11 colunas
meta_alfabetizacao_municipio            10,704 linhas   14 colunas


In [63]:
def analisar_dataframe(nome: str, df: pd.DataFrame) -> None:
    """
    Exibe um resumo geral de um DataFrame da camada Bronze.

    A função apresenta informações básicas sobre a estrutura dos dados,
    incluindo quantidade de registros, quantidade de colunas, tipos de
    dados, valores nulos e registros duplicados.

    Parameters
    ----------
    nome : str
        Nome do dataset utilizado na identificação da análise.

    df : pandas.DataFrame
        DataFrame que será analisado.

    Returns
    -------
    None
        A função apenas exibe informações no notebook.
    """

    print("=" * 80)
    print(nome.upper())
    print("=" * 80)

    print(f"Linhas: {df.shape[0]:,}")
    print(f"Colunas: {df.shape[1]}")

    print("\nTipos de dados")
    display(df.dtypes)

    print("\nValores nulos")
    display(df.isnull().sum())

    print("\nDuplicados")

    print(df.duplicated().sum())

    print("\nPrimeiros registros")

    display(df.head())

In [67]:
def verificar_chave(df: pd.DataFrame, colunas: list[str]) -> int:
    """
    Verifica se um conjunto de colunas pode ser utilizado como chave candidata.

    A função contabiliza a quantidade de registros duplicados considerando
    apenas as colunas informadas.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame a ser analisado.

    chave : list[str]
        Lista contendo as colunas que compõem a chave candidata.

    Returns
    -------
    int
        Quantidade de registros duplicados encontrados para a chave informada.
        Valor igual a zero indica que a chave é candidata.
    """

    qtd = df.duplicated(subset=colunas).sum()

    print(f"Chave: {colunas}")
    print(f"Duplicados: {qtd}")

    return qtd

In [69]:
def resumo_dataframe(
    nome: str,
    df: pd.DataFrame,
    chave: list[str] | None = None
) -> None:
    """
    Exibe um resumo executivo da qualidade do dataset.

    O resumo apresenta informações consolidadas sobre o DataFrame,
    incluindo quantidade de registros, colunas, duplicidades,
    chave candidata e colunas com valores nulos.

    Parameters
    ----------
    nome : str
        Nome do dataset.

    df : pandas.DataFrame
        DataFrame analisado.

    chave : list[str], optional
        Colunas utilizadas para validar a chave candidata.

    Returns
    -------
    None
        A função apenas imprime o resumo da análise.
    """

    print("=" * 80)
    print(nome.upper())
    print("=" * 80)

    print(f"Linhas      : {df.shape[0]:,}")
    print(f"Colunas     : {df.shape[1]}")
    print(f"Duplicados  : {df.duplicated().sum()}")

    if chave:
        duplicados_chave = df.duplicated(subset=chave).sum()
        print(f"Chave candidata ({', '.join(chave)}): {duplicados_chave} duplicados")

    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]

    print("\nColunas com valores nulos")

    if nulos.empty:
        print("Nenhuma")
    else:
        display(nulos.to_frame(name="Quantidade"))

# Análise UF

In [70]:
resumo_dataframe(
    "UF",
    datasets["uf"],
    chave=["ano", "sigla_uf", "serie", "rede"]
)

UF
Linhas      : 145
Colunas     : 16
Duplicados  : 0
Chave candidata (ano, sigla_uf, serie, rede): 0 duplicados

Colunas com valores nulos


,Quantidade
proporcao_aluno_nivel_0,70
proporcao_aluno_nivel_1,70
proporcao_aluno_nivel_2,70
proporcao_aluno_nivel_3,70
proporcao_aluno_nivel_4,70
proporcao_aluno_nivel_5,70
proporcao_aluno_nivel_6,70
proporcao_aluno_nivel_7,70
proporcao_aluno_nivel_8,70


## Conclusões - UF

### Qualidade dos dados

- A tabela possui **145 registros** e **16 colunas**.
- Não foram identificados registros duplicados.
- A combinação (`ano`, `sigla_uf`, `serie`, `rede`) comportou-se como uma **chave candidata**, sem duplicidades.

### Valores nulos

Foram identificados valores nulos apenas nas colunas:

- proporcao_aluno_nivel_0
- proporcao_aluno_nivel_1
- proporcao_aluno_nivel_2
- proporcao_aluno_nivel_3
- proporcao_aluno_nivel_4
- proporcao_aluno_nivel_5
- proporcao_aluno_nivel_6
- proporcao_aluno_nivel_7
- proporcao_aluno_nivel_8

Esses nulos aparentam fazer parte da estrutura original da Base dos Dados e, portanto, **não serão tratados na camada Silver**.

### Decisão para a Silver

- Não remover duplicados.
- Não alterar tipos de dados.
- Não substituir valores nulos.
- Manter a estrutura original da tabela.

# Análise Município

In [71]:
analisar_dataframe(
    "municipio",
    datasets["municipio"]
)

MUNICIPIO
Linhas: 23,995
Colunas: 16

Tipos de dados


ano                          Int64
id_municipio                object
id_municipio_nome           object
serie                       object
rede                        object
taxa_alfabetizacao         float64
media_portugues            float64
proporcao_aluno_nivel_0    float64
proporcao_aluno_nivel_1    float64
proporcao_aluno_nivel_2    float64
proporcao_aluno_nivel_3    float64
proporcao_aluno_nivel_4    float64
proporcao_aluno_nivel_5    float64
proporcao_aluno_nivel_6    float64
proporcao_aluno_nivel_7    float64
proporcao_aluno_nivel_8    float64
dtype: object


Valores nulos


ano                            0
id_municipio                   0
id_municipio_nome              0
serie                          0
rede                           0
taxa_alfabetizacao             0
media_portugues                0
proporcao_aluno_nivel_0    11547
proporcao_aluno_nivel_1    11547
proporcao_aluno_nivel_2    11547
proporcao_aluno_nivel_3    11547
proporcao_aluno_nivel_4    11547
proporcao_aluno_nivel_5    11547
proporcao_aluno_nivel_6    11547
proporcao_aluno_nivel_7    11547
proporcao_aluno_nivel_8    11547
dtype: int64


Duplicados
0

Primeiros registros


,ano,id_municipio,id_municipio_nome,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2023,1100031,Cabixi,2° ano do Ensino Fundamental,Municipal,69.10,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,1100072,Corumbiara,2° ano do Ensino Fundamental,Municipal,58.20,747.8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,1100189,Pimenta Bueno,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),69.73,762.4062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,1101609,Theobroma,2° ano do Ensino Fundamental,Municipal,50.70,745.6802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,1101807,Vale do Paraíso,2° ano do Ensino Fundamental,Municipal,55.69,752.3724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [72]:
verificar_chave(
    datasets["municipio"],
    [
        "ano",
        "id_municipio",
        "serie",
        "rede"
    ]
)

Chave: ['ano', 'id_municipio', 'serie', 'rede']
Duplicados: 0


np.int64(0)

In [73]:
resumo_dataframe(
    "Município",
    datasets["municipio"],
    chave=[
        "ano",
        "id_municipio",
        "serie",
        "rede"
    ]
)

MUNICÍPIO
Linhas      : 23,995
Colunas     : 16
Duplicados  : 0
Chave candidata (ano, id_municipio, serie, rede): 0 duplicados

Colunas com valores nulos


,Quantidade
proporcao_aluno_nivel_0,11547
proporcao_aluno_nivel_1,11547
proporcao_aluno_nivel_2,11547
proporcao_aluno_nivel_3,11547
proporcao_aluno_nivel_4,11547
proporcao_aluno_nivel_5,11547
proporcao_aluno_nivel_6,11547
proporcao_aluno_nivel_7,11547
proporcao_aluno_nivel_8,11547


In [29]:
datasets["municipio"].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ano,23995.0,<NA>,<NA>,<NA>,2023.518775,0.499658,2023.0,2023.0,2024.0,2024.0,2024.0
id_municipio,23995,5550,2902658,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id_municipio_nome,23995,5281,Bom Jesus,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
serie,23995,1,2° ano do Ensino Fundamental,23995,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rede,23995,4,Municipal,10896,NaN,NaN,NaN,NaN,NaN,NaN,NaN
taxa_alfabetizacao,23995.0,NaN,NaN,NaN,61.444545,19.743521,2.12,47.17,62.23,76.55,100.0
media_portugues,23995.0,NaN,NaN,NaN,751.375708,23.232837,673.2983,736.03575,750.6542,764.3029,868.4554
proporcao_aluno_nivel_0,12448.0,NaN,NaN,NaN,2.201809,2.689885,0.0,0.0,1.52,3.28,30.0
proporcao_aluno_nivel_1,12448.0,NaN,NaN,NaN,4.308785,4.296197,0.0,0.83,3.32,6.4925,50.0
proporcao_aluno_nivel_2,12448.0,NaN,NaN,NaN,7.733355,6.313496,0.0,2.8475,6.58,11.39,47.06


In [30]:
datasets["municipio"]["id_municipio"].head(20)

0     1100031
1     1100072
2     1100189
3     1101609
4     1101807
5     1302900
6     1303304
7     1500107
8     1501105
9     1501725
10    1502103
11    1502855
12    1503705
13    1506583
14    1600105
15    1600600
16    1600709
17    1700350
18    1703602
19    1718006
Name: id_municipio, dtype: object

In [31]:
datasets["municipio"]["id_municipio"].apply(type).value_counts()

id_municipio
<class 'str'>    23995
Name: count, dtype: int64

## Conclusões - Município

### Qualidade dos dados

- A tabela possui **23.995 registros** e **16 colunas**.
- Não foram identificados registros duplicados.
- A combinação (`ano`, `id_municipio`, `serie`, `rede`) comportou-se como uma **chave candidata**, sem duplicidades.

### Valores nulos

Foram identificados valores nulos apenas nas colunas:

- proporcao_aluno_nivel_0
- proporcao_aluno_nivel_1
- proporcao_aluno_nivel_2
- proporcao_aluno_nivel_3
- proporcao_aluno_nivel_4
- proporcao_aluno_nivel_5
- proporcao_aluno_nivel_6
- proporcao_aluno_nivel_7
- proporcao_aluno_nivel_8

Esses valores aparentam representar ausência legítima de informação e **serão preservados na camada Silver**.

### Identificador do município

A coluna `id_municipio` é armazenada como **string**.

Após validação, verificou-se que todos os registros utilizam esse tipo de dado.

Como se trata de um identificador (código IBGE) e não de um valor numérico, **o tipo será mantido como string na camada Silver**.

### Decisão para a Silver

- Não remover duplicados.
- Não alterar tipos de dados.
- Não substituir valores nulos.
- Manter `id_municipio` como string.
- Preservar a estrutura original da tabela.

# Análise Meta Alfabetização Brasil

In [32]:
analisar_dataframe(
    "meta_alfabetizacao_brasil",
    datasets["meta_alfabetizacao_brasil"]
)

META_ALFABETIZACAO_BRASIL
Linhas: 3
Colunas: 11

Tipos de dados


ano                          Int64
rede                        object
taxa_alfabetizacao         float64
meta_alfabetizacao_2024    float64
meta_alfabetizacao_2025    float64
meta_alfabetizacao_2026    float64
meta_alfabetizacao_2027    float64
meta_alfabetizacao_2028    float64
meta_alfabetizacao_2029    float64
meta_alfabetizacao_2030    float64
percentual_participacao    float64
dtype: object


Valores nulos


ano                        0
rede                       0
taxa_alfabetizacao         0
meta_alfabetizacao_2024    0
meta_alfabetizacao_2025    0
meta_alfabetizacao_2026    0
meta_alfabetizacao_2027    0
meta_alfabetizacao_2028    0
meta_alfabetizacao_2029    0
meta_alfabetizacao_2030    0
percentual_participacao    0
dtype: int64


Duplicados
0

Primeiros registros


,ano,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao
0,2025,Pública,66.0,60.0,64.00,67.00,71.00,74.00,77.00,80.0,88.00
1,2024,Pública,59.2,59.9,63.77,67.47,70.97,74.23,77.24,80.0,87.37
2,2023,Pública,55.9,59.9,63.77,67.47,70.97,74.23,77.24,80.0,86.00


In [33]:
verificar_chave(
    datasets["meta_alfabetizacao_brasil"],
    [
        "ano",
        "rede"
    ]
)

Chave: ['ano', 'rede']
Duplicados: 0


np.int64(0)

In [34]:
resumo_dataframe(
    "Meta Alfabetização Brasil",
    datasets["meta_alfabetizacao_brasil"],
    chave=[
        "ano",
        "rede"
    ]
)

META ALFABETIZAÇÃO BRASIL
Linhas      : 3
Colunas     : 11
Duplicados  : 0
Chave candidata (ano, rede): 0 duplicados

Colunas com valores nulos
Nenhuma


In [35]:
datasets["meta_alfabetizacao_brasil"].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ano,3.0,<NA>,<NA>,<NA>,2024.0,1.0,2023.0,2023.5,2024.0,2024.5,2025.0
rede,3,1,Pública,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
taxa_alfabetizacao,3.0,NaN,NaN,NaN,60.366667,5.150081,55.9,57.55,59.2,62.6,66.0
meta_alfabetizacao_2024,3.0,NaN,NaN,NaN,59.933333,0.057735,59.9,59.9,59.9,59.95,60.0
meta_alfabetizacao_2025,3.0,NaN,NaN,NaN,63.846667,0.132791,63.77,63.77,63.77,63.885,64.0
meta_alfabetizacao_2026,3.0,NaN,NaN,NaN,67.313333,0.271355,67.0,67.235,67.47,67.47,67.47
meta_alfabetizacao_2027,3.0,NaN,NaN,NaN,70.98,0.017321,70.97,70.97,70.97,70.985,71.0
meta_alfabetizacao_2028,3.0,NaN,NaN,NaN,74.153333,0.132791,74.0,74.115,74.23,74.23,74.23
meta_alfabetizacao_2029,3.0,NaN,NaN,NaN,77.16,0.138564,77.0,77.12,77.24,77.24,77.24
meta_alfabetizacao_2030,3.0,NaN,NaN,NaN,80.0,0.0,80.0,80.0,80.0,80.0,80.0


## Conclusões - Meta Alfabetização Brasil

### Evidências

- A tabela possui **3 registros** e **11 colunas**.
- Não foram encontrados registros duplicados.
- Não foram identificados valores nulos.
- A combinação (`ano`, `rede`) comportou-se como uma chave candidata.
- Todos os tipos de dados encontram-se consistentes com o conteúdo da tabela.

### Decisões para a Silver

- Manter a estrutura original da tabela.
- Não remover registros.
- Não alterar tipos de dados.
- Não aplicar tratamento de valores nulos.
- Preservar todas as colunas de metas (2024 a 2030).

# Análise Meta Alfabetização UF

In [36]:
analisar_dataframe(
    "meta_alfabetizacao_uf",
    datasets["meta_alfabetizacao_uf"]
)

META_ALFABETIZACAO_UF
Linhas: 81
Colunas: 13

Tipos de dados


ano                          Int64
sigla_uf                    object
sigla_uf_nome               object
rede                        object
taxa_alfabetizacao         float64
meta_alfabetizacao_2024    float64
meta_alfabetizacao_2025    float64
meta_alfabetizacao_2026    float64
meta_alfabetizacao_2027    float64
meta_alfabetizacao_2028    float64
meta_alfabetizacao_2029    float64
meta_alfabetizacao_2030    float64
percentual_participacao    float64
dtype: object


Valores nulos


ano                        0
sigla_uf                   0
sigla_uf_nome              0
rede                       0
taxa_alfabetizacao         4
meta_alfabetizacao_2024    9
meta_alfabetizacao_2025    3
meta_alfabetizacao_2026    2
meta_alfabetizacao_2027    2
meta_alfabetizacao_2028    2
meta_alfabetizacao_2029    2
meta_alfabetizacao_2030    2
percentual_participacao    4
dtype: int64


Duplicados
0

Primeiros registros


,ano,sigla_uf,sigla_uf_nome,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao
0,2024,RR,Roraima,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,RR,Roraima,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024,SE,Sergipe,Pública,38.39,38.3,45.9,53.6,61.2,68.3,74.6,80.0,92.84
3,2023,SE,Sergipe,Pública,31.30,38.3,45.9,53.6,61.2,68.3,74.6,80.0,88.34
4,2025,SE,Sergipe,Pública,50.00,38.0,46.0,54.0,61.0,68.0,75.0,80.0,87.00


In [37]:
verificar_chave(
    datasets["meta_alfabetizacao_uf"],
    [
        "ano",
        "sigla_uf",
        "rede"
    ]
)

Chave: ['ano', 'sigla_uf', 'rede']
Duplicados: 0


np.int64(0)

In [38]:
resumo_dataframe(
    "Meta Alfabetização UF",
    datasets["meta_alfabetizacao_uf"],
    chave=[
        "ano",
        "sigla_uf",
        "rede"
    ]
)

META ALFABETIZAÇÃO UF
Linhas      : 81
Colunas     : 13
Duplicados  : 0
Chave candidata (ano, sigla_uf, rede): 0 duplicados

Colunas com valores nulos


,Quantidade
taxa_alfabetizacao,4
meta_alfabetizacao_2024,9
meta_alfabetizacao_2025,3
meta_alfabetizacao_2026,2
meta_alfabetizacao_2027,2
meta_alfabetizacao_2028,2
meta_alfabetizacao_2029,2
meta_alfabetizacao_2030,2
percentual_participacao,4


In [39]:
datasets["meta_alfabetizacao_uf"].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ano,81.0,<NA>,<NA>,<NA>,2024.0,0.821584,2023.0,2023.0,2024.0,2025.0,2025.0
sigla_uf,81,27,RR,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sigla_uf_nome,81,27,Roraima,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rede,81,1,Pública,81,NaN,NaN,NaN,NaN,NaN,NaN,NaN
taxa_alfabetizacao,77.0,NaN,NaN,NaN,59.033117,12.188871,31.3,51.12,59.13,66.7,85.31
meta_alfabetizacao_2024,72.0,NaN,NaN,NaN,58.252778,9.97261,38.0,52.1,57.0,65.25,80.0
meta_alfabetizacao_2025,78.0,NaN,NaN,NaN,62.202564,7.849212,45.9,56.925,61.4,67.3,80.0
meta_alfabetizacao_2026,79.0,NaN,NaN,NaN,66.221899,6.018411,53.6,62.2,65.7,70.1,80.0
meta_alfabetizacao_2027,79.0,NaN,NaN,NaN,70.126076,4.245984,61.0,67.3,70.0,72.9,80.0
meta_alfabetizacao_2028,79.0,NaN,NaN,NaN,73.711519,2.632844,68.0,72.0,73.4,75.25,80.0


In [40]:
datasets["meta_alfabetizacao_uf"]["sigla_uf"].nunique()

27

In [41]:
sorted(
    datasets["meta_alfabetizacao_uf"]["sigla_uf"].unique()
)

['AC',
 'AL',
 'AM',
 'AP',
 'BA',
 'CE',
 'DF',
 'ES',
 'GO',
 'MA',
 'MG',
 'MS',
 'MT',
 'PA',
 'PB',
 'PE',
 'PI',
 'PR',
 'RJ',
 'RN',
 'RO',
 'RR',
 'RS',
 'SC',
 'SE',
 'SP',
 'TO']

In [42]:
datasets["meta_alfabetizacao_uf"][
    datasets["meta_alfabetizacao_uf"].isnull().any(axis=1)
]

,ano,sigla_uf,sigla_uf_nome,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao
0,2024,RR,Roraima,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,RR,Roraima,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,2025,AC,Acre,Pública,68.00,NaN,57.0,62.00,67.00,72.00,76.00,80.0,82.00
22,2024,AC,Acre,Pública,51.38,NaN,56.9,62.20,67.30,72.00,76.20,80.0,80.87
23,2023,AC,Acre,Pública,NaN,NaN,56.9,62.20,67.30,72.00,76.20,80.0,NaN
24,2025,RR,Roraima,Pública,57.00,NaN,NaN,62.33,67.36,72.01,76.24,80.0,82.00
45,2025,DF,Distrito Federal,Pública,65.00,NaN,63.0,67.00,71.00,74.00,77.00,80.0,85.00
48,2023,DF,Distrito Federal,Pública,NaN,NaN,63.1,67.00,70.60,74.00,77.10,80.0,NaN
49,2024,DF,Distrito Federal,Pública,59.13,NaN,63.1,67.00,70.60,74.00,77.10,80.0,79.83


## Conclusões - Meta Alfabetização UF

### Evidências

- A tabela possui **81 registros** e **13 colunas**.
- Não foram encontrados registros duplicados.
- A combinação (`ano`, `sigla_uf`, `rede`) comportou-se como uma chave candidata.
- Foram identificadas exatamente **27 UFs**, indicando cobertura nacional completa.
- Existem valores nulos em algumas métricas e metas futuras.

### Decisões para a Silver

- Manter a estrutura original da tabela.
- Não remover registros.
- Não alterar tipos de dados.
- Preservar os valores nulos, pois sua causa não está documentada e não há evidências suficientes para justificar tratamento.
- Manter `sigla_uf` como chave de integração com a tabela UF.

# Análise Meta Alfabetização Município

In [43]:
analisar_dataframe(
    "meta_alfabetizacao_municipio",
    datasets["meta_alfabetizacao_municipio"]
)

META_ALFABETIZACAO_MUNICIPIO
Linhas: 10,704
Colunas: 14

Tipos de dados


ano                          Int64
id_municipio                object
id_municipio_nome           object
rede                        object
taxa_alfabetizacao         float64
meta_alfabetizacao_2024    float64
meta_alfabetizacao_2025    float64
meta_alfabetizacao_2026    float64
meta_alfabetizacao_2027    float64
meta_alfabetizacao_2028    float64
meta_alfabetizacao_2029    float64
meta_alfabetizacao_2030    float64
nivel_alfabetizacao          Int64
percentual_participacao    float64
dtype: object


Valores nulos


ano                          0
id_municipio                 0
id_municipio_nome            0
rede                         0
taxa_alfabetizacao         120
meta_alfabetizacao_2024    240
meta_alfabetizacao_2025      0
meta_alfabetizacao_2026      0
meta_alfabetizacao_2027      0
meta_alfabetizacao_2028      0
meta_alfabetizacao_2029      0
meta_alfabetizacao_2030      0
nivel_alfabetizacao        120
percentual_participacao    120
dtype: int64


Duplicados
0

Primeiros registros


,ano,id_municipio,id_municipio_nome,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,nivel_alfabetizacao,percentual_participacao
0,2023,4301750,Barão do Triunfo,Municipal,NaN,NaN,14.05,23.65,37.00,52.68,67.85,80.0,<NA>,NaN
1,2024,4301750,Barão do Triunfo,Municipal,4.40,NaN,14.05,23.65,37.00,52.68,67.85,80.0,0,92.59
2,2024,2406908,Lucrécia,Municipal,42.86,7.94,14.05,23.65,37.00,52.68,67.85,80.0,1,84.00
3,2023,2406908,Lucrécia,Municipal,4.40,7.94,14.05,23.65,37.00,52.68,67.85,80.0,0,82.14
4,2023,1718501,Recursolândia,Municipal,4.60,8.25,14.48,24.16,37.49,53.03,68.00,80.0,0,95.65


In [44]:
verificar_chave(
    datasets["meta_alfabetizacao_municipio"],
    [
        "ano",
        "id_municipio",
        "rede"
    ]
)

Chave: ['ano', 'id_municipio', 'rede']
Duplicados: 0


np.int64(0)

In [45]:
resumo_dataframe(
    "Meta Alfabetização Município",
    datasets["meta_alfabetizacao_municipio"],
    chave=[
        "ano",
        "id_municipio",
        "rede"
    ]
)

META ALFABETIZAÇÃO MUNICÍPIO
Linhas      : 10,704
Colunas     : 14
Duplicados  : 0
Chave candidata (ano, id_municipio, rede): 0 duplicados

Colunas com valores nulos


,Quantidade
taxa_alfabetizacao,120
meta_alfabetizacao_2024,240
nivel_alfabetizacao,120
percentual_participacao,120


In [46]:
datasets["meta_alfabetizacao_municipio"].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ano,10704.0,<NA>,<NA>,<NA>,2023.5,0.500023,2023.0,2023.0,2023.5,2024.0,2024.0
id_municipio,10704,5352,4301750,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id_municipio_nome,10704,5096,São Domingos,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rede,10704,1,Municipal,10704,NaN,NaN,NaN,NaN,NaN,NaN,NaN
taxa_alfabetizacao,10584.0,NaN,NaN,NaN,61.774719,19.52754,4.4,47.6,62.445,76.7425,100.0
meta_alfabetizacao_2024,10464.0,NaN,NaN,NaN,62.151745,15.097461,7.94,52.17,64.095,75.885,80.0
meta_alfabetizacao_2025,10704.0,NaN,NaN,NaN,65.245433,12.632025,14.05,57.1075,66.94,76.51,80.0
meta_alfabetizacao_2026,10704.0,NaN,NaN,NaN,68.557849,9.855499,23.65,62.3975,69.875,77.24,80.0
meta_alfabetizacao_2027,10704.0,NaN,NaN,NaN,71.771986,7.060485,37.0,67.3975,72.665,77.95,80.0
meta_alfabetizacao_2028,10704.0,NaN,NaN,NaN,74.794895,4.400614,52.68,72.0375,75.285,78.65,80.0


In [47]:
datasets["meta_alfabetizacao_municipio"]["id_municipio"].nunique()

5352

In [48]:
datasets["meta_alfabetizacao_municipio"]["id_municipio"].apply(type).value_counts()

id_municipio
<class 'str'>    10704
Name: count, dtype: int64

In [49]:
registros_com_nulos = datasets["meta_alfabetizacao_municipio"][
    datasets["meta_alfabetizacao_municipio"].isnull().any(axis=1)
]

print(f"Registros com algum valor nulo: {len(registros_com_nulos)}")
print(f"Percentual: {len(registros_com_nulos) / len(datasets['meta_alfabetizacao_municipio']):.2%}")

registros_com_nulos.head(10)

Registros com algum valor nulo: 240
Percentual: 2.24%


,ano,id_municipio,id_municipio_nome,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,nivel_alfabetizacao,percentual_participacao
0,2023,4301750,Barão do Triunfo,Municipal,NaN,NaN,14.05,23.65,37.00,52.68,67.85,80.0,<NA>,NaN
1,2024,4301750,Barão do Triunfo,Municipal,4.40,NaN,14.05,23.65,37.00,52.68,67.85,80.0,0,92.59
8,2023,2414456,Triunfo Potiguar,Municipal,NaN,NaN,18.14,28.32,41.34,55.70,69.16,80.0,<NA>,NaN
9,2024,2414456,Triunfo Potiguar,Municipal,10.95,NaN,18.14,28.32,41.34,55.70,69.16,80.0,0,80.43
12,2024,2406205,Lagoa d'Anta,Municipal,11.65,NaN,19.26,29.54,42.43,56.43,69.47,80.0,0,74.29
13,2023,2406205,Lagoa d'Anta,Municipal,NaN,NaN,19.26,29.54,42.43,56.43,69.47,80.0,<NA>,NaN
20,2024,2408706,Paraú,Municipal,14.29,NaN,22.17,32.58,45.04,58.16,70.22,80.0,0,70.00
21,2023,2408706,Paraú,Municipal,NaN,NaN,22.17,32.58,45.04,58.16,70.22,80.0,<NA>,NaN
26,2023,1200435,Santa Rosa do Purus,Municipal,NaN,NaN,23.62,34.04,46.27,58.96,70.57,80.0,<NA>,NaN
27,2024,1200435,Santa Rosa do Purus,Municipal,15.62,NaN,23.62,34.04,46.27,58.96,70.57,80.0,0,76.32


## Conclusões - Meta Alfabetização Município

### Evidências

- A tabela possui **10.704 registros** e **14 colunas**.
- Não foram encontrados registros duplicados.
- A combinação (`ano`, `id_municipio`, `rede`) comportou-se como uma chave candidata.
- A coluna `id_municipio` está armazenada como **string**, mantendo consistência com a tabela Município.
- Foram identificados **5.352 municípios distintos**, indicando que nem todos os municípios possuem metas cadastradas.
- Existem valores nulos em `taxa_alfabetizacao`, `meta_alfabetizacao_2024`, `nivel_alfabetizacao` e `percentual_participacao`.
- A distribuição dos nulos indica ausência de informação oficial para determinadas combinações de município e ano, e não erro de ingestão.

### Decisões para a Silver

- Manter a estrutura original da tabela.
- Não remover registros.
- Não alterar tipos de dados.
- Preservar `id_municipio` como string.
- Preservar os valores nulos.
- Utilizar `id_municipio` como chave de integração com a tabela Município.

## Análise Alunos

In [50]:
analisar_dataframe(
    "alunos",
    datasets["alunos"]
)

ALUNOS
Linhas: 3,867,999
Colunas: 13

Tipos de dados


ano                        Int64
id_municipio              object
id_municipio_nome         object
id_escola                 object
id_aluno                  object
caderno                   object
serie                     object
rede                      object
presenca                  object
preenchimento_caderno     object
alfabetizado              object
proficiencia             float64
peso_aluno               float64
dtype: object


Valores nulos


ano                           0
id_municipio                  0
id_municipio_nome             0
id_escola                     0
id_aluno                      0
caderno                       0
serie                         0
rede                          0
presenca                      0
preenchimento_caderno         0
alfabetizado                  0
proficiencia             513338
peso_aluno               513338
dtype: int64


Duplicados
0

Primeiros registros


,ano,id_municipio,id_municipio_nome,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno
0,2023,1304062,Tabatinga,60000459,13059733,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
1,2023,1302603,Manaus,60000712,13047236,1,2° ano do Ensino Fundamental,Estadual,Ausente,Prova não preenchida,Não,NaN,NaN
2,2023,1303403,Parintins,60001028,13053625,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
3,2023,1502806,Curralinho,60003295,15041371,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
4,2023,1500602,Altamira,60003373,15006332,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN


In [59]:
verificar_chave(
    datasets["alunos"],
    [
        "ano",
        "id_aluno"
    ]
)

Chave: ['ano', 'id_aluno']
Duplicados: 0


np.int64(0)

In [52]:
resumo_dataframe(
    "Alunos",
    datasets["alunos"],
    chave=[
        "ano",
        "id_municipio",
        "rede"
    ]
)

ALUNOS
Linhas      : 3,867,999
Colunas     : 13
Duplicados  : 0
Chave candidata (ano, id_municipio, rede): 3855568 duplicados

Colunas com valores nulos


,Quantidade
proficiencia,513338
peso_aluno,513338


In [53]:
datasets["alunos"].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ano,3867999.0,<NA>,<NA>,<NA>,2023.548232,0.497668,2023.0,2023.0,2024.0,2024.0,2024.0
id_municipio,3867999,5548,3550308,107729,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id_municipio_nome,3867999,5279,São Paulo,107729,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id_escola,3867999,42811,60040224,579,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id_aluno,3867999,2352328,43027466,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
caderno,3867999,22,1,250518,NaN,NaN,NaN,NaN,NaN,NaN,NaN
serie,3867999,1,2° ano do Ensino Fundamental,3867999,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rede,3867999,3,Municipal,3432576,NaN,NaN,NaN,NaN,NaN,NaN,NaN
presenca,3867999,2,Presente,3355846,NaN,NaN,NaN,NaN,NaN,NaN,NaN
preenchimento_caderno,3867999,2,Prova preenchida,3354661,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [54]:
datasets["alunos"]["id_municipio"].nunique()

5548

In [55]:
datasets["alunos"]["id_municipio"].apply(type).value_counts()

id_municipio
<class 'str'>    3867999
Name: count, dtype: int64

In [56]:
registros_com_nulos = datasets["alunos"][
    datasets["alunos"].isnull().any(axis=1)
]

print(f"Registros com algum valor nulo: {len(registros_com_nulos)}")
print(f"Percentual: {len(registros_com_nulos) / len(datasets['alunos']):.2%}")

registros_com_nulos.head(10)

Registros com algum valor nulo: 513338
Percentual: 13.27%


,ano,id_municipio,id_municipio_nome,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno
0,2023,1304062,Tabatinga,60000459,13059733,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
1,2023,1302603,Manaus,60000712,13047236,1,2° ano do Ensino Fundamental,Estadual,Ausente,Prova não preenchida,Não,NaN,NaN
2,2023,1303403,Parintins,60001028,13053625,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
3,2023,1502806,Curralinho,60003295,15041371,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
4,2023,1500602,Altamira,60003373,15006332,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
5,2023,2300200,Acaraú,60007232,23070698,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
6,2023,2304400,Fortaleza,60009397,23018304,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
7,2023,2401107,Areia Branca,60009666,24009985,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
8,2023,2408102,Natal,60010392,24005396,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
9,2023,2412609,São Paulo do Potengi,60010523,24024061,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN


In [57]:
datasets["alunos"].head()

,ano,id_municipio,id_municipio_nome,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno
0,2023,1304062,Tabatinga,60000459,13059733,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
1,2023,1302603,Manaus,60000712,13047236,1,2° ano do Ensino Fundamental,Estadual,Ausente,Prova não preenchida,Não,NaN,NaN
2,2023,1303403,Parintins,60001028,13053625,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
3,2023,1502806,Curralinho,60003295,15041371,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
4,2023,1500602,Altamira,60003373,15006332,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN


In [58]:
datasets["alunos"].columns.tolist()

['ano',
 'id_municipio',
 'id_municipio_nome',
 'id_escola',
 'id_aluno',
 'caderno',
 'serie',
 'rede',
 'presenca',
 'preenchimento_caderno',
 'alfabetizado',
 'proficiencia',
 'peso_aluno']

## Conclusões - Alunos

### Evidências

- A tabela possui **3.867.999 registros** e **13 colunas**.
- Não foram encontrados registros duplicados.
- A combinação (`ano`, `id_aluno`) comportou-se como uma **chave candidata**, sem duplicidades.
- A combinação (`ano`, `id_municipio`, `rede`) **não representa a granularidade da tabela**, apresentando 3.855.568 registros duplicados quando utilizada como chave.
- A tabela representa registros individuais de alunos avaliados, sendo a menor granularidade disponível na base de dados.
- A coluna `id_municipio` está armazenada como **string**, mantendo consistência com as demais tabelas do projeto.
- Foram identificados **5.548 municípios distintos**, indicando cobertura nacional da avaliação.
- Os valores nulos concentram-se exclusivamente nas colunas `proficiencia` e `peso_aluno`.
- A inspeção dos registros demonstrou que esses valores nulos estão associados, predominantemente, a alunos ausentes ou que não realizaram a avaliação, caracterizando ausência legítima de informação e não falha na ingestão dos dados.

### Decisões para a Silver

- Manter a estrutura original da tabela.
- Não remover registros.
- Não substituir valores nulos.
- Preservar `id_municipio` como string.
- Utilizar (`ano`, `id_aluno`) como chave candidata da tabela.
- Preservar os registros de alunos ausentes, deixando eventuais filtros para análises específicas na camada Gold.

# Conclusões Gerais

A análise exploratória permitiu concluir que:

- Todos os datasets apresentam boa qualidade.
- Não foram encontrados registros duplicados.
- Todas as tabelas possuem chaves candidatas identificadas.
- Os valores nulos encontrados são compatíveis com a documentação e com o contexto dos dados.
- Não foram identificadas inconsistências de tipos de dados.
- Os identificadores geográficos (`sigla_uf` e `id_municipio`) apresentaram padronização consistente entre as tabelas.

## Decisão para a Camada Silver

Com base nas evidências levantadas, a camada Silver terá como objetivo principal:

- validar a integridade dos dados;
- padronizar os datasets;
- preservar a estrutura original;
- preparar os dados para consumo na camada Gold.

Não serão realizadas alterações que modifiquem o significado dos dados originais.